In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [9]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [11]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(index=index, llm_client=openai_client, instructions=instructions)
assistant.rag("How do I run Ollama locally")
assistant.rag("How do I run Olama locally?")

'To run Olama (likely referring to the course or a tool within the course) locally, you need to be comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. Ensure you document your setup and keep your environment reproducible. No specific instructions for Olama are provided, but the general guidance for running the course locally is given in Module 1: RAG.'

In [12]:
messages = [{"role": "user", "content": "How do I run Ollama locally?"}]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
)
print(response.choices[0].message.content)


<think>
Here's a thinking process:

1.  **Understand User Query:** The user is asking "How do I run Ollama locally?" This is a straightforward technical question about installing and running Ollama, a popular open-source tool for running large language models locally.

2.  **Identify Key Information Needed:**
   - What is Ollama? (Brief context if needed, but focus on the "how")
   - Supported operating systems
   - Installation steps for each major OS (macOS, Windows, Linux)
   - How to verify installation
   - How to run a model locally
   - Basic usage examples
   - Optional: Docker method, system requirements, troubleshooting tips

3.  **Structure the Response:**
   - Prerequisites/System Requirements
   - Installation (macOS, Windows, Linux)
   - Verification
   - Running a Model
   - Basic Usage/Commands
   - Optional: Docker method
   - Tips/Troubleshooting
   - Keep it concise and actionable

4.  **Draft - Section by Section:**
   *(Mental Refinement)*
   - **Prerequisites:** 

In [13]:
messages = [{"role": "user", "content": "How do I run Ollama locally?"}]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)
print(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='x48hxa570', function=Function(arguments='{"query":"run Ollama locally"}', name='search'), type='function')]


In [14]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
results = search(**args)
result_json = json.dumps(results, indent=2)

messages.append({
    "role": "assistant",
    "tool_calls": [{
        "id": tool_call.id,
        "type": "function",
        "function": {
            "name": tool_call.function.name,
            "arguments": tool_call.function.arguments
        }
    }]
})

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json,
})

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)

print(response.choices[0].message.content)

To run Ollama locally, follow these steps based on your operating system:

### 1. Installation
First, download Ollama from [https://ollama.com/download](https://ollama.com/download):

*   **macOS**: Download and install the `.pkg` file.
*   **Windows**: Download and install the `.msi` file.
*   **Linux**: Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Run a Model
Once installed, open your terminal and run the following command to run the **LLaMA 3** model:

```bash
ollama run llama3
```
This command will automatically download the model (approx. 4GB) if you don't have it and start a local chat interface.

### 3. Testing the Server
To verify that the local server is running correctly, you can run:
```bash
curl http://localhost:11434
```
You should receive a JSON response showing the available models.

### 4. Using with Python
If you want to run Ollama via code, install the Python client:
```bash
pip install ollam

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [24]:
def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)
    
    result_json = json.dumps(result, indent = 2)

    return {
        "role": "tool",
        "tool_call_id" : tool_call.id,
        "content": result_json,
    }



In [25]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)

msg = response.choices[0].message
has_function_calls = False

if msg.tool_calls:
    messages.append({
        "role": "assistant",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in msg.tool_calls
        ]
    })
    for tc in msg.tool_calls:
        print("function_call:", tc.function.name, tc.function.arguments)
        call_output = make_call(tc)
        messages.append(call_output)
        has_function_calls = True

elif msg.content:
    print("ASSISTANT:")
    print(msg.content)

function_call: search {"query":"join late"}


In [26]:
def agent_loop(instructions, question, model="qwen/qwen3.6-27b") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append({
                "role": "assistant",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in msg.tool_calls
                ]
            })
            for tc in msg.tool_calls:
                print("function_call:", tc.function.name, tc.function.arguments)
                call_output = make_call(tc)
                messages.append(call_output)
                has_function_calls = True

        elif msg.content:
            print("ASSISTANT:")
            last_answer = msg.content
            print(msg.content)

        it += 1
        if not has_function_calls:
            break

    return last_answer

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Ollama locally install"}
iteration #2...
ASSISTANT:
Based on the course FAQs, here are the steps to install and run Ollama locally:

### 1. Installation

Visit **[https://ollama.com/download](https://ollama.com/download)** and select your operating system:

*   **macOS:** Download the `.pkg` file and follow the installation wizard.
*   **Windows:** Download the `.msi` file and follow the installation wizard.
*   **Linux:** Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Running a Model
Once installation is complete, open your terminal or command prompt and type:
```bash
ollama run llama3
```
This command will download the LLaMA 3 model (approximately 4GB) if you haven't used it before, start it locally, and open a chat-style interface where you can type your questions.

### 3. Testing the Local Server
To verify that Ollama is running as a local server, you can u

'Based on the course FAQs, here are the steps to install and run Ollama locally:\n\n### 1. Installation\n\nVisit **[https://ollama.com/download](https://ollama.com/download)** and select your operating system:\n\n*   **macOS:** Download the `.pkg` file and follow the installation wizard.\n*   **Windows:** Download the `.msi` file and follow the installation wizard.\n*   **Linux:** Run the following command in your terminal:\n    ```bash\n    curl -fsSL https://ollama.com/install.sh | sh\n    ```\n\n### 2. Running a Model\nOnce installation is complete, open your terminal or command prompt and type:\n```bash\nollama run llama3\n```\nThis command will download the LLaMA 3 model (approximately 4GB) if you haven\'t used it before, start it locally, and open a chat-style interface where you can type your questions.\n\n### 3. Testing the Local Server\nTo verify that Ollama is running as a local server, you can use the following curl command:\n```bash\ncurl http://localhost:11434\n```\nIf suc

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join late course"}
iteration #2...
ASSISTANT:
Yes, you can still join! The course materials are available, and you can start whenever you want.

However, please note that if you want to receive a certificate, you need to submit your final project while the submission window is still open. Also, you'll need to participate in the peer-review process during the time the course is actively running with a live cohort, as certificates are not awarded for purely self-paced participation outside of those windows.

To get started, you can check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

Is there anything else you'd like to explore regarding the course materials or homework deadlines?


"Yes, you can still join! The course materials are available, and you can start whenever you want.\n\nHowever, please note that if you want to receive a certificate, you need to submit your final project while the submission window is still open. Also, you'll need to participate in the peer-review process during the time the course is actively running with a live cohort, as certificates are not awarded for purely self-paced participation outside of those windows.\n\nTo get started, you can check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nIs there anything else you'd like to explore regarding the course materials or homework deadlines?"

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...


''

In [30]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
Based on the current course FAQ database, there is no information regarding "Queen Gambit." It appears this topic is not covered in the course materials.

Is there anything else related to the course or its logistics that you would like to explore?


'Based on the current course FAQ database, there is no information regarding "Queen Gambit." It appears this topic is not covered in the course materials.\n\nIs there anything else related to the course or its logistics that you would like to explore?'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I couldn't find any information about "queen gambit" in the course FAQ database. It is likely an off-topic question as it refers to a TV series or chess concept not covered in the course materials.

Is there anything else regarding the course or its logistics that you would like to explore?


'I couldn\'t find any information about "queen gambit" in the course FAQ database. It is likely an off-topic question as it refers to a TV series or chess concept not covered in the course materials.\n\nIs there anything else regarding the course or its logistics that you would like to explore?'